# Biology-Inspired Optimization — Swarm Visualization

This notebook visualizes how **ABC**, **CS**, **FA**, and **PSO** explore a 2-D
fitness landscape (Rastrigin function), showing how agents (bees, cuckoos,
fireflies, particles) move toward the global optimum over iterations.

Each algorithm runs with `stat=True` to record full population snapshots,
then we animate the swarm movement on a contour plot.

In [3]:
pip install git+https://github.com/NgTHung/CSC14003-AIP1@asPackage

  Cloning https://github.com/NgTHung/CSC14003-AIP1 (to revision asPackage) to /tmp/pip-req-build-fn6mzq8b
  Running command git clone --filter=blob:none --quiet https://github.com/NgTHung/CSC14003-AIP1 /tmp/pip-req-build-fn6mzq8b
  Running command git checkout -b asPackage --track origin/asPackage
  Switched to a new branch 'asPackage'
  Branch 'asPackage' set up to track remote branch 'asPackage' from 'origin'.
  Resolved https://github.com/NgTHung/CSC14003-AIP1 to commit 79611279d82ee4c55ce33e6ffa601178b99bf98c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of csc14003-aip1 to determine which version is compatible with other requirements. This could take a while.
ERROR: Package 'csc14003-aip1' requires a different Python: 3.12.12 not in '>=3.14'
Note: you may need to restart th

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))
# If running from the docs/ folder, adjust path:
if not os.path.isdir(os.path.join(sys.path[0], 'problems')):
    sys.path[0] = os.path.join(os.getcwd(), 'src')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from AIP.problems.continuous import Rastrigin
from AIP.algorithm.natural.biology.abc import ArtificialBeeColony, ABCParameter
from AIP.algorithm.natural.biology.cs import CuckooSearch, CuckooSearchParameter
from AIP.algorithm.natural.biology.fa import FireflyAlgorithm, FireflyParameter
from AIP.algorithm.natural.biology.pso import ParticleSwarmOptimization, PSOParameter

print('Imports OK')

ModuleNotFoundError: No module named 'AIP'

## 1. Setup — Problem & Landscape Grid

We use the 2-D **Rastrigin** function (highly multimodal, global min at origin).

In [ ]:
# 2-D Rastrigin
problem = Rastrigin(n_dim=2)

# Grid for contour plot
x_lin = np.linspace(-5.12, 5.12, 200)
y_lin = np.linspace(-5.12, 5.12, 200)
X, Y = np.meshgrid(x_lin, y_lin)
grid = np.column_stack([X.ravel(), Y.ravel()])
Z = problem.eval(grid).reshape(X.shape)

# Quick preview
fig, ax = plt.subplots(figsize=(6, 5))
cf = ax.contourf(X, Y, Z, levels=30, cmap='viridis', alpha=0.8)
ax.set_title('Rastrigin 2-D Landscape')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
plt.colorbar(cf, ax=ax, label='f(x)')
ax.plot(0, 0, 'r*', markersize=15, label='Global min (0,0)')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Run All Four Algorithms (with history recording)

In [ ]:
N_ITER = 50  # iterations for all algorithms
np.random.seed(42)

# --- ABC ---
abc = ArtificialBeeColony(
    ABCParameter(n_bees=25, limit=50, iteration=N_ITER),
    problem, stat=True
)
abc.run()
print(f'ABC  best fitness: {abc.best_fitness:.6f}  at {abc.best_solution}')

# --- CS ---
cs = CuckooSearch(
    CuckooSearchParameter(n_nests=25, pa=0.25, alpha=0.01, beta=1.5, iteration=N_ITER),
    problem, stat=True
)
cs.run()
print(f'CS   best fitness: {cs.best_fitness:.6f}  at {cs.best_solution}')

# --- FA ---
fa = FireflyAlgorithm(
    FireflyParameter(n_fireflies=25, alpha=0.5, beta0=1.0, gamma=1.0,
                     alpha_decay=0.97, cycle=N_ITER),
    problem, stat=True
)
fa.run()
print(f'FA   best fitness: {fa.best_fitness:.6f}  at {fa.best_solution}')

# --- PSO ---
pso = ParticleSwarmOptimization(
    PSOParameter(n_particles=25, w=0.7, c1=1.5, c2=1.5, v_max=1.0, cycle=N_ITER),
    problem, stat=True
)
pso.run()
print(f'PSO  best fitness: {pso.best_fitness:.6f}  at {pso.best_solution}')

## 3. Convergence Curves

Compare how quickly each algorithm's best fitness drops over iterations.

In [ ]:
def best_fitness_over_time(history_snapshots, problem):
    """Given a list of population snapshots, compute running best fitness."""
    best = float('inf')
    curve = []
    for snap in history_snapshots:
        fits = problem.eval(snap)
        gen_best = float(np.min(fits))
        best = min(best, gen_best)
        curve.append(best)
    return curve

abc_curve = best_fitness_over_time(abc.food_sources_history, problem)
cs_curve  = best_fitness_over_time(cs.nests_history, problem)
fa_curve  = best_fitness_over_time(fa.firefly_pos_history, problem)
pso_curve = best_fitness_over_time(pso.particle_position_history, problem)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(abc_curve, label='ABC (Bees)', linewidth=2)
ax.plot(cs_curve,  label='CS (Cuckoos)', linewidth=2)
ax.plot(fa_curve,  label='FA (Fireflies)', linewidth=2)
ax.plot(pso_curve, label='PSO (Particles)', linewidth=2)
ax.set_xlabel('Iteration'); ax.set_ylabel('Best Fitness')
ax.set_title('Convergence Comparison')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Animated Swarm Visualizations

Each animation shows agents moving on the Rastrigin contour map.
- The **colored dots** are individual agents (bees / cuckoos / fireflies / particles).
- The **red star** marks the current global best.
- The title updates with the iteration number and best fitness.

In [ ]:
def animate_swarm(history, agent_name, color, marker='o'):
    """Create a matplotlib animation of a population moving on the landscape.

    Parameters
    ----------
    history : list[np.ndarray]
        List of population snapshots, each of shape (n_agents, 2).
    agent_name : str
        Name shown in the title (e.g. 'Bees', 'Particles').
    color : str
        Matplotlib color for the agent scatter dots.
    marker : str
        Marker style.

    Returns
    -------
    HTML
        Embeddable animation.
    """
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.contourf(X, Y, Z, levels=30, cmap='viridis', alpha=0.7)
    ax.plot(0, 0, 'r*', markersize=15, zorder=5)
    ax.set_xlim(-5.12, 5.12); ax.set_ylim(-5.12, 5.12)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')

    scat = ax.scatter([], [], c=color, edgecolors='white',
                      s=50, zorder=4, marker=marker, linewidths=0.5)
    best_dot, = ax.plot([], [], 'r*', markersize=18, zorder=6)
    title = ax.set_title('')

    def init():
        scat.set_offsets(np.empty((0, 2)))
        best_dot.set_data([], [])
        return scat, best_dot, title

    def update(frame):
        positions = history[frame]  # (n_agents, 2)
        scat.set_offsets(positions[:, :2])
        fits = problem.eval(positions)
        best_idx = int(np.argmin(fits))
        bx, by = positions[best_idx, 0], positions[best_idx, 1]
        best_dot.set_data([bx], [by])
        title.set_text(f'{agent_name}  —  iter {frame+1}/{len(history)}'
                       f'    best f = {float(fits[best_idx]):.4f}')
        return scat, best_dot, title

    anim = FuncAnimation(fig, update, init_func=init,
                         frames=len(history), interval=150, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

### 4.1 ABC — Artificial Bee Colony
Watch the **bees** (food sources) cluster around promising regions and
occasionally scouts jump to new random positions.

In [ ]:
animate_swarm(abc.food_sources_history, 'ABC (Bees)', color='gold', marker='h')

### 4.2 CS — Cuckoo Search
Watch the **cuckoo eggs** jump via Lévy flights — occasionally making
long jumps to explore, with worst nests being abandoned.

In [ ]:
animate_swarm(cs.nests_history, 'CS (Cuckoos)', color='dodgerblue', marker='^')

### 4.3 FA — Firefly Algorithm
Watch the **fireflies** attract each other based on brightness —
brighter (better) fireflies pull dimmer ones closer, forming clusters.

In [ ]:
animate_swarm(fa.firefly_pos_history, 'FA (Fireflies)', color='orange', marker='*')

### 4.4 PSO — Particle Swarm Optimization
Watch the **particles** converge — each particle balances attraction
to its personal best and the global best.

In [ ]:
animate_swarm(pso.particle_position_history, 'PSO (Particles)', color='lime', marker='o')

## 5. Side-by-Side Snapshot Comparison

Static view of all four algorithms at selected iteration snapshots.

In [ ]:
algos = [
    ('ABC (Bees)',      abc.food_sources_history,          'gold',       'h'),
    ('CS (Cuckoos)',    cs.nests_history,                  'dodgerblue', '^'),
    ('FA (Fireflies)',  fa.firefly_pos_history,            'orange',     '*'),
    ('PSO (Particles)', pso.particle_position_history,     'lime',       'o'),
]

snapshots = [0, N_ITER // 4, N_ITER // 2, N_ITER - 1]  # iterations to show

fig, axes = plt.subplots(len(algos), len(snapshots),
                         figsize=(4 * len(snapshots), 4 * len(algos)),
                         sharex=True, sharey=True)

for row, (name, hist, color, marker) in enumerate(algos):
    for col, it in enumerate(snapshots):
        ax = axes[row][col]
        ax.contourf(X, Y, Z, levels=20, cmap='viridis', alpha=0.6)
        pos = hist[it]
        ax.scatter(pos[:, 0], pos[:, 1], c=color, edgecolors='k',
                   s=50, marker=marker, linewidths=0.5, zorder=4)
        ax.plot(0, 0, 'r*', markersize=12, zorder=5)
        ax.set_xlim(-5.12, 5.12); ax.set_ylim(-5.12, 5.12)

        if row == 0:
            ax.set_title(f'Iter {it + 1}', fontsize=12, fontweight='bold')
        if col == 0:
            ax.set_ylabel(name, fontsize=11, fontweight='bold')

fig.suptitle('Swarm Positions at Selected Iterations', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Population Diversity Over Time

Measures the average pairwise distance between agents each iteration.
A rapid drop indicates convergence; sustained diversity means broader exploration.

In [ ]:
def diversity(history):
    """Mean Euclidean distance from population centroid per iteration."""
    divs = []
    for snap in history:
        centroid = snap.mean(axis=0)
        dists = np.linalg.norm(snap - centroid, axis=1)
        divs.append(float(dists.mean()))
    return divs

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(diversity(abc.food_sources_history),      label='ABC', linewidth=2)
ax.plot(diversity(cs.nests_history),               label='CS',  linewidth=2)
ax.plot(diversity(fa.firefly_pos_history),          label='FA',  linewidth=2)
ax.plot(diversity(pso.particle_position_history),   label='PSO', linewidth=2)
ax.set_xlabel('Iteration'); ax.set_ylabel('Mean Distance from Centroid')
ax.set_title('Population Diversity Over Time')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()